<h1 align="center">TÉCNICO EM CIÊNCIA DE DADOS</h1>

<h2 align="center">Roteiro do Desenvolvendo Juntos</h2>

<br>

**Componente:** Análise Exploratória de Dados e Inteligência de Negócios  
**Unidade Curricular:** Análise Exploratória de Dados  
**Tema da Semana:** Transformação e Preparação de Dados  
**Semana 9**  
**Aula 1:** Transformação de dados para análise

<br>


# Prática guiada — Transformar para quê?

**Situação profissional**

Você acompanha, com a equipe de dados, a base integrada de uma pequena rede de papelarias. A investigação já passou por seleção, qualidade e preparação inicial. Agora, algumas perguntas exigem olhar com atenção para **a forma como os dados estão representados e para o que queremos conseguir com eles**.

Nesta prática guiada, o professor conduzirá quatro situações. **A pergunta vem antes do código.** Observe o que temos, acompanhe a decisão tomada, execute a célula quando indicado e compare o que aparece antes e depois.

**Perguntas que vão acompanhar toda a prática**

1. Quando precisamos transformar dados?
2. O que muda quando fazemos uma transformação?
3. Como saber qual transformação faz sentido?
4. Quando não precisamos transformar nada?

Não precisamos responder tudo agora. Vamos observar os casos primeiro.

**Fluxo de trabalho**

**Finalidade → observar → perguntar → executar → comparar → guardar a dúvida**


## 1. Preparação no VS Code

Arquivos usados nesta prática guiada:

- `DADOS3EMC3B2S9A1_desenvolvendo_juntos.ipynb`
- `rede_papelarias_B2_integrado.csv`

1. Salve os dois arquivos **na mesma pasta**.
2. Abra o arquivo `.ipynb` no **Visual Studio Code**.
3. Se solicitado, selecione um **kernel Python** disponível.
4. Execute a célula abaixo para carregar a base.

> Se o professor optar pelo Google Colab, faça o upload do arquivo CSV para a sessão antes de executar a célula.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

dados = pd.read_csv("rede_papelarias_B2_integrado.csv")

# Trabalhamos em cópia e reproduzimos a retirada da cópia integral já confirmada no projeto.
dados_pg = dados.drop_duplicates().copy()

print(f"Registros para a prática: {len(dados_pg)}")
print(f"Colunas disponíveis: {dados_pg.shape[1]}")


## 2. Antes dos casos — conheça as variáveis que vão aparecer

Nesta prática, vamos usar apenas algumas colunas da base:

- `despesas_com_pessoal` — despesa diária registrada com pessoal.
- `entradas` — quantidade de entradas registradas no dia.
- `cupons_emitidos` — quantidade de transações registradas no dia.
- `loja` — identificador da loja.

Os quatro casos usam a **mesma base**, mas fazem perguntas diferentes.


In [ ]:
dados_pg[[
    "id_registro", "loja", "entradas", "cupons_emitidos", "despesas_com_pessoal"
]].head(8)


## Caso 1 — Tenho a informação, mas consigo usá-la?

**Finalidade**

Queremos comparar despesas com pessoal. Antes de qualquer cálculo, observe dois registros da mesma coluna.

Pergunte à turma:

1. Os dois valores parecem representar o mesmo tipo de informação?
2. Se quisermos fazer uma operação numérica com esses valores, há algum obstáculo?
3. O problema parece estar no valor da despesa ou na forma como ele aparece na base?

Execute a célula abaixo apenas depois da discussão.


In [ ]:
caso1 = dados_pg.loc[
    dados_pg["id_registro"].isin(["R00001", "R00693"]),
    ["id_registro", "loja", "despesas_com_pessoal"]
].copy()

caso1


Agora acompanhe a mudança feita na cópia da coluna. O código já está preparado; o foco é observar **antes e depois**.


In [ ]:
texto = dados_pg["despesas_com_pessoal"].astype("string").str.strip()
texto = texto.str.replace("R$", "", regex=False).str.strip()

# Nos registros com vírgula decimal, ajustamos a escrita para a conversão numérica.
usa_virgula = texto.str.contains(",", na=False)
texto.loc[usa_virgula] = (
    texto.loc[usa_virgula]
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)

dados_pg["despesas_com_pessoal_num"] = pd.to_numeric(texto, errors="coerce")

caso1_depois = dados_pg.loc[
    dados_pg["id_registro"].isin(["R00001", "R00693"]),
    ["id_registro", "loja", "despesas_com_pessoal", "despesas_com_pessoal_num"]
].copy()

caso1_depois


In [ ]:
media_dos_dois_registros = caso1_depois["despesas_com_pessoal_num"].mean()
print(f"Média dos dois valores selecionados: R$ {media_dos_dois_registros:.2f}")


**Pare e observe**

- A despesa registrada nos dois casos mudou?
- O que passou a ser possível fazer depois da mudança?
- Guarde a dúvida: **o que exatamente foi transformado neste caso?**


## Caso 2 — Os números já funcionam, mas respondem à pergunta?

**Finalidade**

Queremos comparar o **aproveitamento das entradas** em dois dias da operação.

Observe os registros e responda antes de executar a transformação:

1. Qual deles parece ter melhor resultado se olharmos apenas para `cupons_emitidos`?
2. Os dois dias tiveram o mesmo número de entradas?
3. Comparar apenas 45 com 75 é suficiente para responder à pergunta?
4. Que outra informação já presente nos registros precisa entrar na comparação?


In [ ]:
caso2 = dados_pg.loc[
    dados_pg["id_registro"].isin(["R00009", "R00023"]),
    ["id_registro", "loja", "entradas", "cupons_emitidos"]
].copy()

caso2


Acompanhe a nova medida e compare novamente os dois registros.


In [ ]:
caso2["cupons_por_100_entradas"] = (
    100 * caso2["cupons_emitidos"] / caso2["entradas"]
).round(1)

caso2


**Pare e observe**

- A comparação parece a mesma depois de relacionar cupons e entradas?
- O que conseguimos comparar agora que não estava evidente olhando apenas para o número de cupons?
- Os valores originais tinham algum problema para o computador calcular?
- Guarde a dúvida: **o problema aqui era o mesmo do caso das despesas?**


## Caso 3 — Outra forma de enxergar os mesmos valores

**Finalidade**

Queremos comparar operações que funcionaram em **níveis semelhantes de movimento**.

Primeiro, vamos manter cada quantidade de entradas exatamente como aparece nos registros e observar quantos valores diferentes existem.


In [ ]:
frequencia_valores = dados_pg["entradas"].value_counts().sort_index()

print(f"Valores distintos de entradas: {dados_pg['entradas'].nunique()}")

plt.figure(figsize=(12, 4))
plt.bar(frequencia_valores.index, frequencia_valores.values, width=1.0)
plt.xlabel("Número de entradas")
plt.ylabel("Quantidade de registros")
plt.title("Frequência dos valores individuais de entradas")
plt.tight_layout()
plt.show()


Pergunte à turma:

- O que conseguimos enxergar neste gráfico?
- É fácil identificar grupos de dias com níveis semelhantes de movimento?
- Poderíamos representar esses mesmos valores de uma forma menos detalhada para esta comparação?

Agora acompanhe a criação de faixas. A coluna original `entradas` será preservada.


In [ ]:
limites = [50, 100, 150, 200, 250, 300, 350, 400, 450]
rotulos = [
    "50–99", "100–149", "150–199", "200–249",
    "250–299", "300–349", "350–399", "400–449"
]

dados_pg["faixa_entradas"] = pd.cut(
    dados_pg["entradas"],
    bins=limites,
    labels=rotulos,
    right=False,
    include_lowest=True
)

frequencia_faixas = dados_pg["faixa_entradas"].value_counts().sort_index()

plt.figure(figsize=(10, 4))
plt.bar(frequencia_faixas.index.astype(str), frequencia_faixas.values)
plt.xlabel("Faixa de entradas")
plt.ylabel("Quantidade de registros")
plt.title("Quantidade de registros por faixa de entradas")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
dados_pg.loc[
    dados_pg["entradas"].isin([153, 193, 270]),
    ["id_registro", "loja", "entradas", "faixa_entradas"]
].drop_duplicates(subset=["entradas"]).sort_values("entradas")


**Pare e observe**

- O que ficou mais fácil de enxergar no segundo gráfico?
- O que aconteceu com o nível de detalhe da representação?
- Um dia com 153 entradas deixou de ter 153 entradas?
- O que ganhamos — e o que deixamos de enxergar — quando passamos dos valores individuais para as faixas?

Guarde a dúvida: **por que representar os mesmos dados de maneiras diferentes pode ajudar uma análise?**


## Caso 4 — E se a informação já atender ao que precisamos?

**Finalidade**

Queremos comparar os resultados das oito lojas.

Antes de pensar em qualquer transformação, observe como a variável `loja` já aparece na base.


In [ ]:
pd.DataFrame({"loja": sorted(dados_pg["loja"].dropna().unique())})


Pergunte à turma:

1. A variável já permite identificar os grupos que queremos comparar?
2. Poderíamos trocar `L1`, `L2`, `L3`... por outra representação?
3. Fazer isso acrescentaria alguma informação necessária para esta pergunta?
4. Precisamos transformar só porque uma transformação é possível?

Neste caso, **não execute nenhuma transformação**. A decisão também faz parte da investigação.


## Retome os quatro casos

Sem buscar ainda uma regra geral, registre apenas o que aconteceu em cada situação:

| Caso | O que queríamos fazer | O que fizemos | O que observamos depois |
|---|---|---|---|
| 1 | Trabalhar numericamente com despesas | Mudamos a forma de representar os valores | Uma operação numérica pôde ser executada |
| 2 | Comparar aproveitamento das entradas | Relacionamos cupons e entradas em uma nova medida | A comparação entre os dois dias mudou |
| 3 | Comparar níveis semelhantes de movimento | Organizamos valores individuais em faixas | Os grupos ficaram mais visíveis, com menos detalhe na nova representação |
| 4 | Comparar as oito lojas | Mantivemos a variável como estava | A identificação dos grupos já atendia à pergunta |

Volte às perguntas do início:

- Quando precisamos transformar dados?
- O que muda quando fazemos uma transformação?
- Como saber qual transformação faz sentido?
- Quando não precisamos transformar nada?

**Temos agora quatro decisões diferentes diante dos dados. Precisamos entender o que há por trás delas.**
